# Hong Kong Ghost Signs

## Imports, Config, and Constants

In [ ]:
import os
import json
from random import choice
from datetime import date

# Datetime manipulation
from datetime import datetime
import datefinder

# Dataframe manipulation
import polars as pl
from polars.exceptions import ComputeError
from polars.datatypes import Datetime, String

# Google Maps and Translate API Clients
import googlemaps
from googletrans import Translator
from google.cloud import translate as google_translate

# Generate UUID
import fastnanoid

# GIS Distance Calculation and Projection Transformation
import haversine as hs
from haversine import Unit
from pyproj import Transformer


# Setup Google Translator
translator = Translator()

# Setup Projection Transformer
# WGS 84 (EPSG:4326) to Hong Kong 1980 Grid (EPSG:2326)
transformer = Transformer.from_crs(4326, 2326)

# Configure Polars 
cfg = pl.Config()
cfg.set_tbl_rows(32)
cfg.set_fmt_str_lengths(256)
cfg.set_tbl_width_chars(1000)

# Define Hong Kong Bounding Box
HKBBOX = (22.5639, 114.4013, 22.1771, 113.8373) 

# PATHS to CSVs
DATASRC = "data/source/hkghostmaps-gmaps-export-20220901.csv"
DATASRC_NANOID = "data/cache/hkghostmaps-gmaps-20221011+nanoid.csv"
DATADEST = f"data/export/hkghostmaps-gmaps-prepared-{datetime.isoformat(datetime.now())[:10]}.csv"
JSONDEST = f"data/export/hkghostmaps-gmaps-prepared-{datetime.isoformat(datetime.now())[:10]}.json"

## Load Data
### Generate NanoIDon first load

In [ ]:
if not os.path.isfile(DATASRC_NANOID):
    # Load Features
    df = pl.read_csv(
        source=DATASRC,
        # Prefix columns with 'raw'
        new_columns=[
            'rawX',
            'rawY',
            'rawName',
            'rawDesc',
            'rawGxml'
        ]).filter(
            # Accept Missing Values
            pl.col("rawX").eq(0) | (
            # Filter Non-Hong Kong Markers
            (HKBBOX[3] <= pl.col("rawX")) &
            (pl.col("rawX") <= HKBBOX[1]) &
            (HKBBOX[2] <= pl.col("rawY")) &
            (pl.col("rawY") <= HKBBOX[0]))
        ).sort(by=('rawX','rawY'))

    # Add UUID
    def gen_nano_id(v):
        return fastnanoid.generate(size=12)
    
    df = df.with_columns(
        id = pl.col('rawX').map_elements(gen_nano_id, return_dtype=str)
    )

    df.write_csv(DATASRC_NANOID)
else:
    df= pl.read_csv(
        DATASRC_NANOID,
        new_columns=[
            'rawX',
            'rawY',
            'rawName',
            'rawDesc',
            'rawGxml',
            'id'
        ]
    )

# Text Parsing

## Parse "rawName"
### Identify the type of data and language in 'name' field
- Coordinates
- Address
- All Chinese
- Mixed Chinese & English
- All English

In [ ]:
df = df.with_columns(
    rxNameIsCoordinates=pl.col('rawName').str.contains(r'^(\(22|22)'),
    rxNameIsAddress=pl.col('rawName').str.contains(r' Rd$| St$| Rd W$| Rd E$| Road$| Street$| Strand W$|Rd - Yuen Long$| Ln$'),
    rxNameIsAllChinese=(
        pl.col('rawName').str.contains(r'[\u4e00-\u9fff]+')
    ) & ~(pl.col('rawName').str.contains(r'[a-zA-Z]+')),
    rxNameIsChineseAndEnglish=(
        pl.col('rawName').str.contains(r'[\u4e00-\u9fff]+')
    ) & (pl.col('rawName').str.contains(r'[a-zA-Z]+'))
)

In [ ]:
# In all other cases, treat the 'name' field to be English
df = df.with_columns(
    rxNameIsAllEnglish=(
        ~(df['rxNameIsCoordinates']) &
        ~(df['rxNameIsAddress']) &
        ~(df['rxNameIsAllChinese']) &
        ~(df['rxNameIsChineseAndEnglish'])
    )
)

## Set English Names from `EN` and mixed `EN/ZH-HANT` 'rawName' 

#### Set English from pure English

In [ ]:
df = df.with_columns(
    pl.when(pl.col("rxNameIsAllEnglish"))
    .then(pl.col('rawName').str.to_titlecase().str.strip_chars())
    .otherwise(None).alias('name'),
    pl.when(pl.col("rxNameIsAllEnglish"))
    .then(False)
    .otherwise(None).alias('nameGen'),
)

#### Set English from Mixed English and Chinese

In [ ]:
original = df.filter(df['rxNameIsChineseAndEnglish'])['rawName'].to_list()
translations = ["'Ltd. Co.' Under Newer 'Gospel Nursing Home' Sign",
"Illegible Blue Sign on Tile 'Center'",
"Prince 'Prince Edward' Jewelery",
"Sign Behind Air Conditioners Above 'Tai Hung Roast Goose'",
"'Chengdelong' Rice Shop",
"'Zhonghuayang' Illegible",
"HK.Fruit 'Global High-quality Ingredients (Fruits, Seafood, Hairy Crabs)",
"'College' Faded on the 6th Floor.",
"Side of 'Zuiqionglou' Hotel",
"Namkok 'South Point' (Cafe)"]

mixed_to_en = dict(zip(original,translations))

df = df.with_columns(
    pl.when(
        df['rxNameIsChineseAndEnglish']
    ).then(
        pl.col('rawName').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # ~Human translated
    pl.when(
        df['rxNameIsChineseAndEnglish']
    ).then(
        False
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [ ]:
# Result, with unprocessed rows
# df.select('name','nameGen').head(10)   

In [ ]:
# Result, only processed rows
# df.filter(df['nameGen'] == False).select('name').head(10)

## Set Traditional Chinese Names from `ZH-HANT` 'rawName'

In [ ]:
df = df.with_columns(
    pl.when(
        pl.col("rxNameIsAllChinese")
    ).then(
        pl.col('rawName').str.strip_chars()
    ).otherwise(
        None
    ).alias('name__zh-hant'),
    pl.when(
        pl.col("rxNameIsAllChinese")
    ).then(
        False
    ).otherwise(
        None
    ).alias('nameGen__zh-hant'),
)

In [ ]:
# Result, only processed rows
# df.filter(df['nameGen__zh-hant'] == False).select('name__zh-hant').head(10)

## Translate Traditional Chinese Names to English

`TODO` : Consider moving the API over to Microsoft Translate

- Read [Comparison](https://translatepress.com/microsoft-translator-vs-google-translate/)
- Review [Docs](https://learn.microsoft.com/en-us/azure/ai-services/translator/reference/v3-0-languages)

### Google Translate

In [ ]:
original = [x for x in df['name__zh-hant'].to_list() if x]
translations = [f"'{translator.translate(x, src='zh-tw', dest='en').text}'" for x in original] 

In [ ]:
zhhant_to_en = dict(zip(original,translations))

for src, dest in zhhant_to_en.items():
    print(f'{src} --> {dest}')

In [ ]:
df = df.with_columns(
    pl.when(
        df['rxNameIsAllChinese']
    ).then(
        pl.col('rawName').replace_strict(zhhant_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # Machine translated
    pl.when(
        df['rxNameIsAllChinese']
    ).then(
        True
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [ ]:
# Result, with unprocessed rows
# df.select('name','nameGen','rawName').head(10)   

## Parse "rawDesc"
### Identify the type of data and language in 'description' field
- Names (where missing `name`)
- Descriptions (where `name` exists too)
- Capture Date of the photo
- Image URL
- All Chinese
- Mixed Chinese & English
- All English

#### `imageUrl` and `visitableAsOf`

In [ ]:
def extract_date(s):
    try:
        res = list(datefinder.find_dates(s, base_date=datetime(2023,1,1)))[0]
        return res
    except TypeError:
        return None
    except ComputeError:
        return None
    except IndexError:
        return None

pattern_img_elem = r"<img\b[^>]*>"
pattern_desc_img_url=r'src\s*=\s*"([^"]+)"'
pattern_desc_date=r'Captured'
pattern_desc_stripping_img_url_and_date = r"<br><br>(.*?)\s*Captured"
pattern_labeled_date=r"(Captured|Documented|Spotted)\s+\w+\s+\w*\s*(2022|2023|2024)\.?\W?"


# Parse raw name to identify the data entered in the 'description' field 
df = df.with_columns(
    # Image URLs
    rxImgUrlExtracted=pl.col('rawDesc').str.extract(pattern_desc_img_url),
    rxDescIsImgHtml=pl.col('rawDesc').str.contains(pattern_desc_img_url),
    # Capture date
    rxDescIscapturedDate=pl.col('rawDesc').str.contains(pattern_desc_date),
    rxvisitableAsOfExtracted=pl.col('rawDesc').map_elements(extract_date, return_dtype=Datetime),
    # Copy description for multi-step processing
    # ryDesc_before=pl.col('rawDesc').str.replace(pattern_img_elem,"").str.replace_all("<br>",""),
    ryDesc=pl.col('rawDesc').str.replace(pattern_img_elem,"").str.replace_all("<br>","").str.replace(pattern_labeled_date,"").str.strip_chars().replace("", None),
)

In [ ]:
# Result, only Image URL
# df.filter(pl.col('rxDescIsImgHtml'))['rxImgUrlExtracted', 'rawDesc']

# Result, only Mixes English / Chinese
# df.filter(pl.col('rxDescIscapturedDate'))['rxvisitableAsOfExtracted','rawDesc']

In [ ]:
# BEFORE and AFTER inspection
# pl.concat(
#     (df['ryDesc_before'].alias('BEFORE').to_frame(),
#      df['ryDesc'].str.replace(pattern_labeled_date,"").alias('AFTER').to_frame()
#     ),how='horizontal')

In [ ]:
# Manual
manual_desc_encoding = {
    "Corner of Wing Fung & Queen's Road East. Captured while hoarding was up for new development in late April 2023." : "Captured while hoarding was up for new development in late April 2023."
}

df = df.with_columns(
    pl.when(
        pl.col('ryDesc').is_in(list(manual_desc_encoding.keys()))
    ).then(
        pl.col('ryDesc').replace_strict(manual_desc_encoding, default=None)       
    ).otherwise(
        pl.col('ryDesc')
    )
)

#### Classsify Languages

In [ ]:
df = df.with_columns(
    ryDescIsAllChine=(
        pl.col('ryDesc').str.contains(r'[\u4e00-\u9fff]+')
    ) & ~(pl.col('ryDesc').str.contains(r'[a-zA-Z]+')),
    ryDescIsChineseAndEnglish=(
        pl.col('ryDesc').str.contains(r'[\u4e00-\u9fff]+')
    ) & (pl.col('ryDesc').str.contains(r'[a-zA-Z]+')
        )
)

# In all other cases, treat the 'description' field to be English
df = df.with_columns(
    ryDescIsAllEnglish=(
        df['ryDesc'].is_not_null() &
        ~(df['ryDescIsAllChine']) &
        ~(df['ryDescIsChineseAndEnglish'])
    )
)

In [ ]:
# Result, All Chinese
# df.filter(pl.col('ryDescIsAllChine'))['ryDesc']
# Result, Mixed English / Chinese
# df.filter(pl.col('ryDescIsChineseAndEnglish'))['ryDesc']
# Result, All English
# df.filter(pl.col('ryDescIsAllEnglish'))['ryDesc']

## Determine which All English descriptions can be promoted to the English name

In [ ]:
desc_to_name_exceptions = [
    "Revealed sign after a car accident broke off a chunk of the building.",
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory.",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete."
]

desc_to_name_exception_names = {
    "Revealed sign after a car accident broke off a chunk of the building." : "Revealed By Car Accident",
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory." : "Ghost Sign Trio",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete." : "Dragon Phoenix Photography Studio",
    "'同' above Healthy Wood on the side next to garage" : "Together"
}

desc_to_name_exception_descs = {
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory." : "Trading Company, Plumber and Electrician, Garment Factory",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete." : "Along with an unreadable sign remnant on concrete",
    "(one day before it was scheduled to be painted over)" : "The day before it was scheduled to be painted over",
    "寶 something trading - vacant shop with for lease sign over the shop front" : "Vacant shop with for lease sign over the shop front",
    "金堂大廈 sign under air conditioning units over 3 shop" : "Under air conditioning units over 3 shop",
    '億和公司 revealed after vegetable stall closed'  : "Revealed after vegetable stall closed",
    '地產代理 in white in side alley.': "In white, found in a side alley",
    '灸 next to Stern Rockwell.' : "Next to Stern Rockwell",
    'Duplicate jeans sign of the one being covered down the road Po Sang Trading 寶生易公司' : "Duplicate jeans sign of the one being covered down the road Po Sang Trading",
    '中國肉公司 - behind roll up gate.' : "Behind a roll up gate",
    '陳董仁 painted over in yellow on Reclamation Street' : "Painted over in yellow on Reclamation Street",
    "'智' on corner of Nanking and Shanghai Street" : "On corner of Nanking and Shanghai Street",
    "Chinese Medicine and '牛王' red letters on blue tile" : "Red letters on blue tile",
    "'同' above Healthy Wood on the side next to garage" : "Above Healthy Wood on the side next to garage"
}

df = df.with_columns(
    pl.when(
        (pl.col('name').is_null()) &
        (pl.col('ryDescIsAllEnglish')) &
        (pl.col('name').is_null()) &
        ~(pl.col('ryDesc').is_in(desc_to_name_exceptions))
    ).then(
        pl.col('ryDesc').str.to_titlecase().str.strip_chars().str.replace(r'$\(','').str.replace(r'\)^','')
    ).otherwise(
        pl.col('name')
    ).alias(
        'name'
    ),
    # Set Name Generated Field to False
    pl.when(
        (pl.col('name').is_null()) &
        (pl.col('ryDescIsAllEnglish')) &
        (pl.col('name').is_null()) &
        ~(pl.col('ryDesc').is_in(desc_to_name_exceptions))
    ).then(
        False
    ).otherwise(
        pl.col("nameGen")
    ).alias('nameGen'),
    # Set Description Field to None -- silly code
    pl.when(
        True      
    ).then(
        None
    ).otherwise(
        None
    ).alias('description'),   
    # Set Description Generated Field to False
    pl.when(
        True
    ).then(
        None
    ).otherwise(
        None
    ).alias('descriptionGen')   
)

# Manual Correction of Names
df = df.with_columns(
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_names.keys()))
    ).then(
        pl.col('ryDesc').replace_strict(desc_to_name_exception_names, default=None)  
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_names.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

# Manual Correction of Descriptions
df = df.with_columns(
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_descs.keys()))
    ).then(
        pl.col('ryDesc').replace_strict(desc_to_name_exception_descs, default=None)    
    ).otherwise(
        pl.col('description')
    ).alias('description'),
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_descs.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('descriptionGen')
    ).alias('descriptionGen'),
)


## Determine which Mixed English / Chinese descriptions can be promoted to the English name

In [ ]:
original_still_missing_name = df.filter((df['name'].is_null()) & (df['ryDescIsChineseAndEnglish']))['ryDesc'].to_list()

translations = [
    "'Precious' Something Trading",
    "'Jintang Building'",
    "'Yihe Company'",
    "'Estate agency'",
    "'Moxibustion'",
    "Behind 'Cheng Hing' and 'Defa Tea Restaurant'",
    "Multiple 'Runfa' Signs",
    "'Jeans'",
    "Illegible 'Company'",
    "Side of building 'Country _ Fixed Head'",
    "'Big Welding Supply Co.'",
    "'Yajiu Engineering Co. Ltd.'",
    "'China Meat Company'",
    "'Chen Dongren'",
    "'Wisdom'",
     "'West Branch', then Illegible",
    "Chinese Medicine and 'Ox King'",
   "'Together'",
 ]

mixed_to_en = dict(zip(original_still_missing_name,translations))

df = df.with_columns(
    pl.when(
       (df['name'].is_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        pl.col('ryDesc').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # ~Human translated
    pl.when(
       (df['name'].is_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        False
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [ ]:
# Results, processed rows
# df.filter((df['name'].is_not_null()) & (df['ryDescIsChineseAndEnglish']))['name','description', 'ryDesc']

## Determine which Mixed English / Chinese descriptions can be translated and used as the English description

In [ ]:
original_already_have_names = df.filter(
        (df['nameGen'] != False) & 
        (df['name'].is_not_null()) & 
        (df['ryDescIsChineseAndEnglish'])
    )['ryDesc'].to_list()

translations = [
 "Above dessert place",
 "In the back alley behind Ping Pong",
 "In yellow",
 "Illegible 'hardware' - behind car repair shop sign",
 "Double layer with paint peeling off to reveal 'Lu Apartment Shanghai'",
 "'Lin_Wen` Chinese Doctor and 2 or 3 more illegible to the right",
 "'Irobu' painted over sign - off Jordan Road",
 "'Peace' under signage from old Standard Chartered Branch"
]

mixed_to_en = dict(zip(original_already_have_names,translations))

df = df.with_columns(
    pl.when(
        (df['nameGen'] != False) & (df['name'].is_not_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        pl.col('ryDesc').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('description')
    ).alias('description'),
    # ~Human translated
    pl.when(
       (df['nameGen'] != False) & (df['name'].is_not_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        False
    ).otherwise(
        pl.col('descriptionGen')
    ).alias('descriptionGen'),
)

In [ ]:
# Results, processed rows
# df.filter((df['description'].is_not_null()) & (df['ryDescIsChineseAndEnglish']))['id','name','description','ryDesc']

In [ ]:
old_ids = ["oRF5wDrHYk6Uoa2JeUjQg2",
"HS392WkyEvD8HM2Xb8FMUQ",
"fGM9VAHMzm4MreR4YoDVRX",
"GkBYLjuBgu7CGQYBcMogXs",
"DvAff69iG3Jhn84gwVobiA",
"2u9Dq3woi5Kco3byib93mZ",
"QiV7p6KQTwZxscrdxfV9eX",
"Udih7uaLLYsjEr5HVJu3Ga",
"WJAuYEZcwcvmqHzPtLDpZE",
"igpwZHtigwAuaFbMdBsmA5",
"Z5MDPMP5wXeFFjZYddbFGU",
"8FktfduLzg8TfFWVqbpLnN",
"nvTWmeTDAtDN8Jjnwgbptg",
"mCsNPSt8ppEwAZL8u4xhJA",
"X6PaudpDaDXShsmVMbH6JG",
"8wTe5j6hzPJdFPsk3cK9Tv",
"ZyUwCkBFha2kPGWreXWFsK",
"bEM8ZadZ2hwUbmGiS7XZP5",
"5j4sEBTPRX2CR5f5qiPK2K",
"FvzJVkBA5MJALhNFMAvZEf",
"bnGThpZxHhfsUJNeaPzKin",
"jeNsuEwF7Sqyij6uqctzeY",
"a3vcMhgk26WLU8qVPpFbHf",
"9a6rNyDgnQCDydHP7F5jgP",
"C5pWAh5SvptK5JkboBzyRH",
"HnoBqyDPF3cXRBDemkFHqq",
"bh6SxbrradWqhiSh6wxRhm",
"ASH3cGdGZZyZGPrTqp5KQu",
"fwkb5R596BSfZJgms7jU6g",
"f9ECqRa5FHP7rCRXEBxtpe",
"AaVLsZw6r5rJkx8bYTY9VJ",
"mvvt8cdRA6YoRf8oTWjHAs",
"5xKYgNtsKvrdfdFTXXswm5",
"3PqQiL8AUAqMLLgfNnufU2",
"CPE49czkq3ExN8AzWuP3qB",
"7avn3DxQ5oPNjTYq7LhNPa",
"MoUenBfFNz8kwiC9cYv7xE",
"SXzbKw562zzZMGxrMeD2b2",
"B4zefs24vB4dwMqtETg9Uf",
"g75tway7j5FHkAcRMWb9M9",
"h6H8bHYcNCMRNu6JQZtZas",
"aJuJLctRuEGN3fVsPthMZA",
"JSrHippq8sn6487Soy57Uj",
"3dnJVzRqv2qcfiHUrfZx6v",
"6gNePfh82C5hbHwK765qTR",
"57zYT4juezARukNoPUvpWF",
"c8wBf9ut8XX7y5soxDFrqB",
"nyT9EJVDdrHqQXhdhPkeYr",
"3z2uW4oPgrj6dkK2Cnem5z",
"nA8YGAJVBFYFYAW7LqFYp6",
"CmVNihXCjM2TL6R6NDdrS7",
"mWFQstFZ7EHYZoa4UoF4vb",
"m7NhfixXRzNyPtdZfscfHu",
"dQAb9HheBKb456PhbQXwTN",
"nzAhQWc66HPtsif9nfffeU",
"95pnmn2DrzQ4PCqAUS3kMS",
"JfgYZT53QvQDgAVheDXhJU",
"b5xgzdeidUsg7edwKBCBx9",
"jiroSpEvyrcx2Wwk5Ng4xh",
"YsZZHso25jpZZiu7Vaqwtg",
"LptoTzneUZUCZP5m67vHwo",
"jGUB3WKsiTc2rLei52uJYr",
"Xtf28aBiQf3WjaNRAKUMwK",
"LaGV2gE4KvwbE2s9nV7dpC",
"39PWBCY8oaCgsN5bkpdU3y",
"bjDWVmrswSBcXtWBxxgfMx",
"fzMrED3D4axY4XeMnwqhpm",
"EKbpgZVzMQy5o5zsfSgggC",
"e9q8cJuzExQgdDqUhUa7e9",
"FVWJmux7NTVM5qurPZBaAS",
"5N32PCzYi4ayYQuYhoe2cx",
"LpcT8Kdv6fSEGjpKhoX5Hq",
"m3Qx24RTsmr7tqj3wiDxu9",
"NUgzuRjNHhicgTBA7hhwEJ",
"J9bmrw9ezZZcCsoDyZ5LSR",
"YSJAgQ3KBFi9PCi8stWVVx",
"7PcNbuZcSYwjP2QixeT3wu",
"HbK5ZHbDgP5LHoTdk95deX",
"2i4TTX64YTPwXPyhsN2Fsm",
"FRo7s44ZATrTu8iaF4T3fH",
"oVV4fz7T6uFpvMURNPuB2J",
"7D3Ah4TFXVGGkPmbdMRWeX",
"eYommpJE7SNovRGvqiZ6b9",
"ZjRRGVqWj7GfUyqBWCSwZg",
"i3DoWD43iBazJdYR77sUT7",
"GhconHB9XwJBJbNFX3tvtx",
"2HzzoaVSqTc5oojUHeG2Ng",
"PrhRzZGd3Q3WPUqTZzAaPD",
"P2j8oMbUYMvooxMJaAvGz2",
"UvkwaoBNQ2D4fr5miC8zay",
"GFxTA6aKWaKD2rWWYrhQBA",
"5CGpNw9S78bvcXHCB5TyXv",
"QuehBcGBNtEdhjUEkYNuH4",
"aw7wLFXS6gYrP9sv63Sfok",
"GRARP8Q9YhUMZpybV4bHwX",
"YaSgzj7yePaGtELNgE7MJZ",
"8ERjiZK7KmGxi4b79yZFsu",
"JfkXm4dkeVL3n7rJnCJiaH",
"mgywFz3T2PG2YGwiUnMxhH",
"QciME7KFd4hikrcza5LwqE",
"BeKzddE95gW5TFfVYdg9ex",
"8pitV8kTSXWubMu9Xyi55e",
"CNdfJwoyxrBGzEi3P84mVC",
"AVq3zf3tShowV8rvAeoMLn",
"hFoGNtT7nb2xWwp7hodrSn",
"BEUDiomL7x8k56hixQaJH2",
"2HT3nqSVitBoFJKwgsLGj9",
"PSK7PGRAmDguipUhdSxTLH",
"mVtgmye4mxXT3SPArCRpfM",
"gqjJmgyv9vvcssNYebDxgC",
"mCEPQKUwFCy8cYZkjuRbBf",
"jueyvJ3o9mVo7QbQpEDFCV",
"J3ZgttCaPZV8STErD5Rec4",
"ercLbAFBV8m3hqGHtTMWna",
"fFLpYi6t9TL3VG62JRzaDg",
"KyAjMAaG7hJegaN3mh4ejX",
"NEuqXq2g5z38T59i5PV9UJ",
"9nemgCes6G3a6qQZkHm3Et",
"mQqpDuZ2N68yaDCs6LHTDr",
"XEMmsYKgCPHAYQYbTEkctH",
"XubM5v5rizLXPenqHVVBCh",
"U7PDZpzB9HCYAaP2tM6BAw",
"aQbyZbHtM6oKqCpDarPnSi",
"AGqTMMDUJJyKhdDfDx6GVJ",
"mBvX5RnjZwhrT5NC6cmFyY",
"TjSqbaRW8REbwjiQWkqyR7",
"crp6v9ZsF74VZyYj6U5YTz",
"oM3ZjnLFLLTzhkotSMubHa",
"8FuWiXL9X4hZny3vnAfkqg",
"U3uF9cWfArVW93rRzMCpi2",
"V8tLxg2b9aXRjngTiourjR",
"gksoiqGpNsDjkKQLk5tb84",
"Vot2dzdDMSL66B5BPcnJhM",
"8ggceZU2StHvXGiWHLmRkD",
"TRMbX4TvRe444sHHuSPDNv",
"L8wk9HKUmSsurAEGnfeevs",
"4eKvWmhs8YuBWtX4XgXJsv",
"bA3Xh4sivm59KjQMow5j8w",
"n2ixChFqwNiS2Pcs9SRy7r",
"2ycZtqk8wPZ2BiRM8HCgkB",
"cDhw2vewTHSXfE8eqFzAzv",
"5Bsb54bHEtbmin53iJ2LcG",
"j2nwKLW4e8JrbttCfTmHZB",
"KRp7Vr4kNRBQEpssQMoHZi",
"GMR22xxHhJU3WsZyJrzwNx",
"GVVsGygNKZtVXxz2fPHUZ4",
"7dFfj8vVzee6ZaTHBLCEVg",
"JZyfThUaX9ff4qAgMfDTL2",
"UvpjQjVjRg8AotKbsghMxW",
"dDbDsVYT5uXbwjbJnAv7W3",
"N677gMJWaTrBP4FFnPtpNE",
"QZUqzCJBXdKL2RghAJGMM7",
"5BjhirrwnxxRFKR2MZ3zPB",
"KSqHT4BpdqyRq3wDpTkY5M",
"kW5VwJjTw8SFsT8kL8DV4d",
"XBqS3iUXzTDG4afBekbuoc",
"4QEUmAmDHbXxohLBbewewT",
"PDzT2NrMLnwbC3ny4rT3go",
"ccbfgPACppuLUYWZwXPv8v",
"A28srmuiGMSKxWCFwFCY8r",
"B2rwxEp4rxk3YY8AnH8b5f",
"CxqgN7g2bQVojz3yFcDhKF",
"fwALnwHJYzTDWrigX2vVdR",
"AjbyWfK6Lo5U55NkxpHcxv",
"W86vHa4vkbW7VPDxoHWZnc",
"gmyBUNrK6gARB9qP5gm3E8",
"FxNSjY87vFVWGoh2Drxn8M",
"LmARjD9DUGQsThp72RBpgM",
"GwQo7JvMcV5Yzwy7oWFpyf",
"8Fzdz5XmLY5kURSFUbu3Hx",
"AXPkkwLhy7SoMYAFEs7KLu",
"oD662jsYCKGmZxGq9mwAdj",
"fcZRQyNo7xaQstr7AMDqwu",
"mQuVFwmXsvw2sojziVZRUF",
"gyhvpZ2WybEDgwBBzD7QqC",
"YWbEY5jdn5QqRaa5qzAYKD",
"Sm2WszcqZcNLWXJU8MMv2A",
"YMHpHSi94LQm69pz9PSYaj",
"oGoev2mBpKKCu82CYNqGKQ",
"9PxBa983ke63zw82mbj8ec",
"HbeQK884oJfuzYibSBiGQX",
"JuxA3HQde3Gi4WdrsEyEsN",
"axrHTxU4wm8KHPCCKFdu9F",
"C7ZHHFjsZ3WutrrcKPthbo",
"DR8cqN4JvzHzSdQL6ep3td",
"GYDfzBCidoGGG6MSjANBQB",
"43sibmZgVfzC3jxFYF3G9J",
"8LYDGku2ADk2TewUGsnTSE",
"mU5anMhPLuiqTv3mCiuV3M",
"QCN4iupKYs5pRDNrouFnHV",
"89wSz7ypF6DnXz6t4oHDXo",
"VNNQMqspEqQqgG3tpwEN4J",
"8ygzaQgTTXVkwRzWg6M7pm",
"jpFw4mN9Ptds3Uan3p48vW",
"68vrhno5Uyv6KxcoSCKxCh",
"T8JfDT2XPTgtkUaGnLu658",
"J2CbvoXJ4Bjs4ZbxFCLF6a",
"VarP8WoK8QeeAbpWZGBU5Z",
"MX8DV7Wqgc6h3dcazj3Joo",
"3fEhXgh9E9ex8H8i8R348U",
"LsoQyq9GB42cssrG9cLAnd",
"FsMKzxCztb8Ld77DrKsygX",
"WdZfsnPNDqJyX4eKDqqrCc",
"RrowtWu8sGmnFRJ5KdwSZ4",
"GJbiRDACW9sPtM4tWVnPuS",
"njaaZeuJrrbSxuKUQCmqf9",
"AGUTNsakGi6od8uJGnMYUY",
"j956oYZo2DZfVMAjHwVTpa",
"3JKb4j42JLDxuevKwqLRJR",
"YYutJAMPGGbDfC29ATAzXC",
"c5fkpvgwTH9XanvXxawjjQ",
"h2tw4jFNCFLMZENhV6btKf",
"JSbNxtJpr7JGN9DPrGHqbj",
"7xpeTGX8e39kmpwtiqBfDx",
"WmvUv39MXQCHCnCTaCt5mU",
"4AaqoLF2jFdKjxXcz64cdx",
"WWmnVh6kySJEL4rd86yakg",
"CSVDQbTAn3v4desnEmQ79q",
"JwbTbdE66jK5Jmx7Khbw35",
"e9Nib53CPtQRqqNdEq7pn7",
"GpsJbbyWN88JNyCxDWsWio",
"67EuJoQx82LdQPfu7wsYnr",
"UyuFy5cAJr7UBZYYMYgbXf",
"UeCrWEV8rrBVbEi6AYMM6R",
"GRuCDuCRQ4a96mTgFFArqh",
"gydmFUJJ9wsVEXbMDc3JUL",
"Hitytg5g62bR7VtEcNJ4gj",
"N7ZfQGiYdwd5LnqbJ8Sgne",
"msyVccn24izGQAPRKxML9c",
"4W6i9WCUf6gusc4VSqaQaT",
"MD4MKjVygZvTXX3XxCKZCJ",
"iQiNrRCipqN22W3Cvgpo2t",
"KPCHKjMVoKYHNGVsvaM83w",
"b7Xvf7yZr28dazZmZ7fnmp",
"ABeLRcfWEBDaoEm3VpJPZo",
"Zte73qxJd6SHJzBfBmxcPL",
"bBcpeUadH2h2cQfnpcN5Av",
"fD2uZpKUfuWDqnsaDyRAdb",
"nBjTa4t8bWVyaZakcbQk2Y",
"jW8J9t3aobEXVQiqLzw4As",
"cAQgPYXLGXNPP8TRC243ku",
"Eu5gBHGa7c4AWo4f7fp7Lg",
"eFJ3Exsxok9mjneqPhbPXX",
"fvtkJFVAVSaH54b9qjDXTL",
"ibyph2LeTXb6uakaeXYPUn",
"CCSBCSJhLJYtxGWKe5FNtL",
"ZYm8arguNX2XCTsfJiSQbd",
"JxvwbtpfrVd3UiJ4jdcLf3",
"LdbWvxiUva5NXrm47Ltj2q",
"W5ZY4TR9jkcCofohRBoAaE",
"hxW4Qp4ymPvvWq3fumdqqQ",
"YsyWsiZcUdajY8TTDruAQE",
"hSXr8WSEXsBwSaw6ufDwvw",
"NSJf9DPT9UmNhonQ7R7DAp",
"jcHxx3gKy5TadiYEnPeg7B",
"hX6FRMrNEsriUrNP47o2FH",
"XHFVcu2yJ6NWxWMbkNb6qU",
"BBhT4TJh6sgXRsnk7SYHRZ",
"UpyR7M3c2TihkjoQqLCCiv",
"g2ZdYYNWmbsLLGBpKeV2VT",
"BXpDeeWMQBkfWoEW9kwrqd",
"RUvaQZSE5774GXzXvVe6Qx",
"SBcteecCLNBMSf83J4mSD8",
"L8CAYq6zyMGReK48yk8ppC",
"8qB6Qn9ntpTf3uyg3stcpr",
"5o6L4gsQonXRPzuxvY4EoH",
"o3ViGpJnXQ46aZt9ooYFrd",
"Czi8oDy7NuZwAtrLhPFkXP",
"X9gx9hSA5mZYGmivFxeNFi",
"m7iTdHxNT8rTHT6nmYfiTv",
"mXTx8X4c3MjcitsacAkGsR",
"BBK5yNM3QjPzZFYW574fMc",
"8RZVHjeyfEFHdA4UDERrfD",
"eYNHkEMYHUgFqyKZMDRaQW",
"kVguXiht7Jcg9juaCRUTBm",
"9cgoXAdCerXPVchjqUqhuw",
"8k6B92p9ju5DtiotmSeDZu",
"XxtDZVUc4urnwRqNhwTiku",
"gRJkhEs7D3NUEbMCYQyVtP",
"mdWcGmkN5nSJmRqgpjbEKY",
"EqnAdiS4MT2eCQRGeaXxVY",
"kaf6GyVnM5SFccoEqRNuWX",
"8JaRxjgpUx3uuGdMSAiuKK",
"f27QMLhTP8NEke4ViiM8iZ",
"RuqLg2tJURTRTxg3yk6YDV",
"TYy8TmtcM6qF7WD2SCoVbr",
"PBSKjgnXyyWYNwHa66WR7o",
"hb59ugoiunzZ2w7CpB7AFh",
"69nLrgP6f98WBF5fx6tAtV"]
id_mapping = dict(zip(old_ids, df['id'].to_list()))

In [ ]:
# Manual correction / cleanup
name_for_id = {
    'fYZgCvPoEgX-': "'Fuhua Tea Shop'",
 'u_YCs9EhIzW2': "'Ming Hua Noodle House'",
 '_oF8Wp_0zaFL': "'He Kee Hardware'",
 'wPGq1HjGOI9P': "'Pan Ji Wood Paint'"
}

In [ ]:
df = df.with_columns(
    pl.when(
        df['id'].is_in(list(name_for_id.keys()))
    ).then(
        pl.col('id').replace_strict(name_for_id, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name')
)

## Set Traditional Chinese names from `ZH-HANT` 'rawDesc'

In [ ]:
df = df.with_columns(
    pl.when(
        (pl.col("ryDescIsAllChine")) &
        (pl.col('name__zh-hant').is_null())
    ).then(
        pl.col('ryDesc').str.strip_chars().str.strip_chars('.')
    ).otherwise(
        pl.col('name__zh-hant')
    ).alias('name__zh-hant'),
    
    pl.when(
        (pl.col("ryDescIsAllChine")) &
        (pl.col('name__zh-hant').is_null())
    ).then(
        False
    ).otherwise(
        pl.col('nameGen__zh-hant')
    ).alias('nameGen__zh-hant')
)

In [ ]:
# Result, only processed rows
# df.filter((pl.col("ryDescIsAllChine")) & (df['nameGen__zh-hant'] == False)).select('name__zh-hant')

## Determine which Mixed English / Chinese descriptions can be used as the Traditional Chinese name

`TODO`:

- Fix the ID references

In [ ]:
mapping_id_to_name_zh_hant = {
 'mZE2eZP2OuPq': "'寶__'",
 'Qf7md_MT4Tqz': "'金堂大廈'",
 '0m5rq24gTjd_': "'億和公司'",
 'l1tFZsddXYFg': "'成德隆'",
 'zQadJjHfXcaT': "'地產代理'",
 '5p0UZrzYsAlw': "'灸'",
 'pgavMGuQmEqU': "'成興和德發茶餐廳'",
 'cJRT4TSLwx61': "'潤發行'",
 'kH3o7jd9PRm5': "'寶生易公司'",
 'zj9HYUyUtBRd': "難以辨認'公司'",
 'NcNv5ESYfVWo': "'國_定頭'",
 'V6Jgz9QBdF-U': "'中華洋'難以辨認",
 'OQocBkRne7Zs': "'大溶接器材行'",
 'fI1Dew2OS1zy': "'雅就工程有限公司'",
 'SR_Gz4NDVs6H': "'陳董仁'",
 'c0fFdQzPo1Uk': "'智'",
 '5iJQSsfq3cXL': "'西可'",
 '0dwg6WdKNKMP': "'牛王'",
 'oF_WSwh-sSGW': "'學院'",
 'TWL4eGFgY1ac': "'醉瓊楼飯店'",
 'MPh1CG2u2ewu': "'尖尖照相'",
 'jx4kIO7pEqxn': "'龍'不完整的",
 'mMt9UC43tmwM': "'同'"}

In [ ]:
df = df.with_columns(
    pl.when(
        df['id'].is_in(list(mapping_id_to_name_zh_hant.keys()))
    ).then(
        pl.col('id').replace_strict(mapping_id_to_name_zh_hant, default=None)
    ).otherwise(
        pl.col('name__zh-hant')
    ).alias('name__zh-hant'),
    
    pl.when(
        df['id'].is_in(list(mapping_id_to_name_zh_hant.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('nameGen__zh-hant')
    ).alias('nameGen__zh-hant')
)

In [ ]:
# Result, processed rows
# df.filter(
#    (df['name__zh-hant'].is_not_null()) &  
#    (df['ryDescIsChineseAndEnglish'])
# )['id','name__zh-hant','nameGen__zh-hant','ryDesc']

## Promote Traditional Chinese descriptions from `rawDesc` to description, if they differ from the Traditional Chinese Name

In [ ]:
df = df.with_columns(
    pl.when(
        pl.col('rxNameIsAllChinese') &
        pl.col('ryDescIsAllChine') &
        ((pl.col('name__zh-hant')) != (pl.col('ryDesc').str.strip_chars('.')))
    ).then(
        pl.col('ryDesc')
    ).otherwise(
        None
    ).alias('description__zh-hant'),
    # NOT Machine translated
    pl.when(
        pl.col('rxNameIsAllChinese') &
        pl.col('ryDescIsAllChine') &
        ((pl.col('name__zh-hant')) != (pl.col('ryDesc').str.strip_chars('.')))
    ).then(
        False
    ).otherwise(
        None
    ).alias('descriptionGen__zh-hant'),
)

In [ ]:
# Result, 
# df[
#  'name',
#  'nameGen',
#  'name__zh-hant',
#  'nameGen__zh-hant',
#  'description',
#  'descriptionGen',
# 'description__zh-hant',
# 'descriptionGen__zh-hant',
#  'rawName',
#  'ryDesc',
# 'rxNameIsAllChinese',
#  'ryDescIsAllChine'
# ].filter(
#     pl.col('rxNameIsAllChinese') &
#     pl.col('ryDescIsAllChine') &
#     ((pl.col('name__zh-hant')) != (pl.col('ryDesc').str.strip_chars('.')))
# )


## Translate English names from `ZH-HANT` 'rawDesc' 

### Google Translate

In [ ]:
original_still_missing_name = df.filter(
    (df['name'].is_null()) & 
    (df['ryDescIsAllChine'])
)['ryDesc'].to_list()

translations = [f"'{translator.translate(x, src='zh-tw', dest='en').text}'" for x in original_still_missing_name]

zhhant_to_en = dict(zip(original_still_missing_name,translations))

for src, dest in zhhant_to_en.items():
    print(f'{src} --> {dest}')

In [ ]:
df = df.with_columns(
    pl.when(
        (df['ryDescIsAllChine'])&
        (df['name']).is_null()
    ).then(
        pl.col('ryDesc').replace_strict(zhhant_to_en, default=None).str.to_titlecase()
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # Machine translated
    pl.when(
        df['ryDescIsAllChine']
    ).then(
        True
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [ ]:
# Result, only processed rows
# df.filter(
#     (df['ryDescIsAllChine']) &
#     (df['nameGen'])
# ).select('name','nameGen','ryDesc')

## Translate English descriptions from `ZH-HANT` 'rawDesc' 

In [ ]:
original_already_have_names_but_missing_desc = df.filter(
        (df['name'].is_not_null()) &
        (df['nameGen'] != True) & 
        (df['description'].is_null()) & 
        (df['ryDescIsAllChine'])
    )['ryDesc'].to_list()

Since this is a null set, no further processing is required.

## Set `visitableAsOf` date

In [ ]:
df = df.cast({pl.Datetime: pl.Date})
df = df.with_columns(
    visitableAsOf=pl.col('rxvisitableAsOfExtracted')
)

## Set `imageUrl` 

In [ ]:
df = df.with_columns(
    imageUrl=pl.col('rxImgUrlExtracted')
)

## Review Parsing Results

In [ ]:
col_order = [
'id',

'name',
'nameGen',
'name__zh-hant',
'nameGen__zh-hant',

'description',
'descriptionGen',
'description__zh-hant',
'descriptionGen__zh-hant',

'rawName',
'ryDesc',

'visitableAsOf',
'imageUrl',

'rxvisitableAsOfExtracted',
'rxImgUrlExtracted',

'rxNameIsAllEnglish',
'rxNameIsAllChinese',
'rxNameIsChineseAndEnglish',
'rxNameIsCoordinates',
'rxNameIsAddress',

'ryDescIsAllEnglish',
'ryDescIsAllChine',
'ryDescIsChineseAndEnglish',
'rxDescIsImgHtml',
'rxDescIscapturedDate',

'rawDesc',
'rawX',
'rawY',
'rawGxml'
]

df = df.select(col_order)

In [ ]:
# df[
#     'name',
#     'nameGen',
#     'name__zh-hant',
#     'nameGen__zh-hant',
#     'description',
#     'descriptionGen',
#     'description__zh-hant',
#     'descriptionGen__zh-hant',
#     'rawName',
#     'ryDesc',
#     'rxNameIsAllChinese',
#     'ryDescIsAllChine'
# ].filter(
#     # pl.col('name').is_null()
#     True
# )

# Geocoding

## Fill missing Latitude & Longitude coordinates

Because the Dataframe is sorted by Latitude, the rows with missing values show up first with a `0` value. There are five of them.

In [ ]:
# Addresses for GhostSigns with missing coordinates
df['rawName'].to_list()[:5]

In [ ]:
original = df.filter(df['rawX'] == 0)['rawName'].to_list()
encodings = [
    (22.328725033185375, 114.18915132846354),
    (22.330166285778667, 114.19210082344293),
    (22.32950210461055, 114.19222392266883),
    (22.32935639108943, 114.1922340848458),
    (22.324942897750564, 114.18932768694137)
]

lats = [lat for (lat,lng) in encodings]
lngs = [lng for (lat,lng) in encodings]

address_to_lat = dict(zip(original,lats))
address_to_lng = dict(zip(original,lngs))

df = df.with_columns(
    pl.when(
        df['rawX'] == 0
    ).then(
        pl.col('rawName').replace_strict(address_to_lng, default=None)
    ).otherwise(
        pl.col('rawX')
    ).alias('longitude'),
    pl.when(
        df['rawY'] == 0
    ).then(
        pl.col('rawName').replace_strict(address_to_lat, default=None)
    ).otherwise(
        pl.col('rawY')
    ).alias('latitude'), 
)

In [ ]:
# Result, with unprocessed rows
df.select('name','longitude','latitude').head(10)   

## Generator the Northing / Easting values in reference to the HK Grid 1980 System

In [ ]:
# Setup Projection Transformer
# WGS 84 (EPSG:4326) to Hong Kong 1980 Grid (EPSG:2326)
transformer = Transformer.from_crs(4326, 2326)

# Apply the projection transformer to all coordinates and
# save into new "northing" and "easting" columns
df = df.with_columns(
    hk_1980_grid = pl.struct(
        ['latitude','longitude']
    ).map_elements(
        lambda coords: transformer.transform(
            coords['latitude'], coords['longitude']
        ), return_dtype=pl.List(pl.Float64)
    ).list.to_struct(fields=['northing','easting'])
).unnest('hk_1980_grid')

## Reverse Geocoding - Get Address from Coordinates

### Setup Google Maps client with our API Key

In [ ]:
env_vars = {}
with open('../.env') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        key, value = line.strip().split('=', 1)
        env_vars[key] = value

gmaps = googlemaps.Client(env_vars['PUBLIC_GOOGLEMAPS_KEY'])

### Define how the Google Maps API response will be used in our data schema

In [ ]:
geoDB = {}

# Mapping from Google API address component name, to our component names
component_prefix = {
    'administrative_area_level_1' : 'administrativeAreaLevel1',
    'country' : "country",
    'neighborhood': "neighbourhood",
    'premise' : "premise",
    'route' : 'route',
    'street_number': 'streetNumber',
    'plus_code' : "plusCode",
    'subpremise' : 'subPremise',
    'intersection' : 'intersection'
}

# Template for Address properties (Formatted Address and Components in EN, ZH-HANT and ZH-HANS, and Metadata)
AddressProperties = {
  "distanceFromPoint" : None,
  'displayAddress' : None,
  'displayAddress__zh-hant' : None,
  'displayAddress__zh-hans' : None,
  'displayAddressGen' : True,
  'displayAddressGen__zh-hant' : True,
  'displayAddressGen__zh-hans' : True,
  "formattedAddress" : None,
  "formattedAddress__zh-hant" : None,
  "formattedAddress__zh-hans" : None,
  "plusCode" : None,
  "plusCode__zh-hant" : None,
  "plusCode__zh-hans" : None,
  "subPremise" : None,
  "subPremise__zh-hant" : None,
  "subPremise__zh-hans" : None,
  "premise" : None,
  "premise__zh-hant" : None,
  "premise__zh-hans" : None,
  "streetNumber" : None,
  "streetNumber__zh-hant" : None,
  "streetNumber__zh-hans" : None,
  "route" : None,
  "route__zh-hant" : None,
  "route__zh-hans" : None,
  "neighbourhood" : None,
  "neighbourhood__zh-hant" : None,
  "neighbourhood__zh-hans" : None,
  "administrativeAreaLevel1" : None,
  "administrativeAreaLevel1__zh-hant" : None,
  "administrativeAreaLevel1__zh-hans" : None,
  "country" : None,
  "country__zh-hant" : None,
  "country__zh-hans" : None,
  "googlePlaceId" : None,
  "geocoder" : None,
  "reverseGen" : False,
  "forwardGen" : False,
} 

def parse_to_address_properties(id, result, src_latlng:tuple, lang="en", geocoder='googlemaps', reverse_encode=True):
    lang_suffix = {
        "en": "",
        "zh-hk": "__zh-hant",
        "zh-cn": "__zh-hans",
    }.get(lang.lower())


    address_properties = AddressProperties.copy() if id not in geoDB else geoDB[id]

    if not len(result) > 0:
        raise IndexError
    
    result = result[0]
    
    for comp in result['address_components']:
        for t in comp['types']:
            if t in ['political','transit_station','bus_station','tourist_attraction', 'point_of_interest', 'establishment', 'park']:
                continue
            field = f'{component_prefix[t]}{lang_suffix}'
            address_properties[field] = comp['long_name']
            
    field = f'formattedAddress{lang_suffix}'
    address_properties[field] = result["formatted_address"]

    field = f'displayAddress{lang_suffix}'
    address_properties[field] = result["formatted_address"]
    
    field = 'distanceFromPoint'
    latlng = result["geometry"]["location"]
    address_properties[field] = int(hs.haversine(
        src_latlng,
        (latlng['lat'], latlng['lng']),
        unit=Unit.METERS
    ))
    
    field = "googlePlaceId"
    address_properties[field] = result["place_id"]

    field = "geocoder"
    address_properties[field] = geocoder

    field = "reverseGen"
    address_properties[field] = reverse_encode

    field = "forwardGen"
    address_properties[field] = reverse_encode == False

    address_properties['id'] = id
    
    geoDB[id] = address_properties

### Request Google Maps API for each LAT,LNG coordinate, and parse the result to Address Properties

We cache the API response so we can use that on subsequent runs

In [ ]:
for row in df.select('id','rawName','latitude','longitude').rows(named=True):
    print(f"GEOFEATURE :: {row['id']} :: {row['rawName']}")
    
    # Look up an address with reverse geocoding
    for lang in ['en','zh-HK','zh-CN']:
        JSONPATH = f'data/cache/{row['id']}-{lang}-gmaps-reverse-geocoded.json'
        if not os.path.isfile(JSONPATH):
            reverse_geocode_result = gmaps.reverse_geocode(
                (row['latitude'], row['longitude']),
                result_type=[
                    'subpremise',
                    'premise',
                    'establishment',
                    'street_number',
                    'street_address',
                    'point_of_interest',
                    'parking',
                    'colloquial_area',
                    'sublocality',
                    'plus_code'

                ],
                location_type=['ROOFTOP','RANGE_INTERPOLATED','GEOMETRIC_CENTER'],
                language=lang
            )

            with open(JSONPATH, 'w', encoding='utf-8') as f:
                json.dump(reverse_geocode_result, f, ensure_ascii=False, indent=4)
        else:
            # Open and read the JSON file
            with open(JSONPATH, 'r') as file:
                reverse_geocode_result = json.load(file)
        
        parse_to_address_properties(row['id'], reverse_geocode_result, (row['latitude'], row['longitude']), lang=lang)

In [ ]:
# Example result
# geoDB['Czi8oDy7NuZwAtrLhPFkXP']

### Merge all Address Properties with Dataframe

In [ ]:
df_geocoded = pl.from_dicts(list(geoDB.values()))
df = df.join(df_geocoded, on="id")

In [ ]:
# Example result
# df.filter(pl.col('id')=='Czi8oDy7NuZwAtrLhPFkXP')

## Forward Geocoding - Get Address Components from Address

The idea is that the address that was originally captured in the dataset is likely to be more accurate than the reverse geocoded address obtained from the coordinates, so we will overwrite the Address Properties of those records where an address was provided.

### Request Google Maps API for each 'rawName' that was an address, and parse the result to Address Properties

We cache the API response so we can use that on subsequent runs

In [1]:
geoDB = {}

for row in df.select('id','rawName','rxNameIsAddress', 'latitude', 'longitude').rows(named=True):
    
    # Only encode raw names which were also addresses
    if not row['rxNameIsAddress']:
        continue
        
    print(f"GEOFEATURE :: {row['id']} :: {row['rawName']}")
    
    # Look up an address with forward geocoding
    for lang in ['en', 'zh-HK', 'zh-CN']:
        JSONPATH = f'data/cache/{row['id']}-{lang}-gmaps-forward-geocoded.json'
        
        if not os.path.isfile(JSONPATH):
            geocode_result = gmaps.geocode(
                row['rawName'],
                components={"country":"HK"},
                language=lang
            )

            with open(JSONPATH, 'w', encoding='utf-8') as f:
                json.dump(geocode_result, f, ensure_ascii=False, indent=4)
        else:
            # Open and read the JSON file
            with open(JSONPATH, 'r') as file:
                geocode_result = json.load(file)
                
        parse_to_address_properties(row['id'], geocode_result, (row['latitude'], row['longitude']), lang=lang, reverse_encode=False)

NameError: name 'df' is not defined

### Define the merge logic for how the new Address Properties should overwrite the columns in the Dataframe

In [ ]:
# The new address components have an intersection component, so we need to backfill this into our original Dataframe
df = df.with_columns(
    pl.lit(None).alias('intersection'),
    pl.lit(None).alias('intersection__zh-hant'),
    pl.lit(None).alias('intersection__zh-hans'),
)

In [ ]:
def update(self:pl.DataFrame, df_other:pl.DataFrame,  join_columns:list[str]) -> pl.DataFrame:
    '''Updates dataframe with the values in df_other after joining on the join_columns'''
    
    # The columns that will be updated
    columns = [c for c in df_other.columns if c not in join_columns]
    
    df_merged = (self
        .join(df_other, how='left', on=join_columns, suffix='_NEW')      
        .with_columns(**{
            c: pl.coalesce([pl.col(c+'_NEW'), pl.col(c)]) for c in columns # <-This updates the columns
        }).select(
            pl.all().exclude('^.*_NEW$') # <- this drops the temporary '*_NEW' columns
           )
       )    
    return df_merged

### Merge the Forward Encoded Address Properties with Dataframe

In [ ]:
df_forward_encoded = pl.from_dicts(list(geoDB.values()))
# df.join(df_forward_encoded, on="id", how='left', coalesce=True)
df = update(df, df_forward_encoded, join_columns=['id'])

# Exporting

In [ ]:
df = df.rename({
    "name" : "title",
    'name__zh-hant' : 'title__zh-hant',
    'nameGen' : 'titleGen',
    'nameGen__zh-hant' : 'titleGen__zh-hant'
})

## Review Result

In [ ]:
col_order = [
'id',

'title',
'title__zh-hant',
'titleGen',
'titleGen__zh-hant',
    
'description',
'description__zh-hant',
'descriptionGen',
'descriptionGen__zh-hant',

'rawName',
'ryDesc',

'longitude',
'latitude',
'distanceFromPoint',

'displayAddress',
'displayAddress__zh-hant',
'displayAddress__zh-hans',
'displayAddressGen',
'displayAddressGen__zh-hant',
'displayAddressGen__zh-hans',
    
'formattedAddress',
'formattedAddress__zh-hant',
'formattedAddress__zh-hans',

'visitableAsOf',
'imageUrl',

'plusCode',
'plusCode__zh-hant',
'plusCode__zh-hans',
'subPremise',
'subPremise__zh-hant',
'subPremise__zh-hans',
'premise',
'premise__zh-hant',
'premise__zh-hans',
'streetNumber',
'streetNumber__zh-hant',
'streetNumber__zh-hans',
'intersection',
'intersection__zh-hant',
'intersection__zh-hans',
'route',
'route__zh-hant',
'route__zh-hans',
'neighbourhood',
'neighbourhood__zh-hant',
'neighbourhood__zh-hans',
'administrativeAreaLevel1',
'administrativeAreaLevel1__zh-hant',
'administrativeAreaLevel1__zh-hans',
'country',
'country__zh-hant',
'country__zh-hans',

'googlePlaceId',
'geocoder',
'reverseGen',
'forwardGen',

'rxvisitableAsOfExtracted',
'rxImgUrlExtracted',

'rxNameIsAllEnglish',
'rxNameIsAllChinese',
'rxNameIsChineseAndEnglish',
'rxNameIsCoordinates',
'rxNameIsAddress',

'ryDescIsAllEnglish',
'ryDescIsAllChine',
'ryDescIsChineseAndEnglish',
'rxDescIsImgHtml',
'rxDescIscapturedDate',

'rawDesc',
'rawX',
'rawY',
'rawGxml'
]

df = df.select(col_order)

In [ ]:
cfg.set_tbl_rows(70)

df.row(44, named=True)

In [ ]:
# cfg.set_tbl_rows(10)
# df.select('name','name__zh-hant','longitude','latitude', 'formattedAddress','formattedAddress__zh-hant','formattedAddress__zh-hans')

## Save Result

In [ ]:
df.write_csv(DATADEST)

### Provisional Data for App

In [ ]:
# df_temp.columns

In [ ]:
df_temp = df

### Fill in missing English names

In [ ]:
df_temp = df_temp.with_columns(
    pl.when(
        pl.col('title').is_null()    
    ).then(
        pl.col('rawName')
    ).otherwise(
        pl.col('title')
    ).alias(
        'title'
    )
)

## Fill `graphemes` with Traditional Chinese title

In [ ]:
df_temp = df_temp.with_columns(
    pl.when(
        pl.col('title__zh-hant').is_not_null()    
    ).then(
        pl.col('title__zh-hant')
    ).otherwise(
        None
    ).alias(
        'graphemes'
    )
)

In [ ]:
selected_cols = [not (x.startswith('raw') | x.startswith('rx') | x.startswith('ry')) for x in df_temp.columns]

In [ ]:
df_temp[selected_cols].write_json(JSONDEST)

In [ ]:
# for k in df_temp[selected_cols].to_dicts()[0].keys():
#     print(k)

In [ ]:
tijptjikId = "p6WnJ-DKl0c1"
features = []
for r in df_temp[selected_cols].to_dicts():
    features.append(
        {
            "id" : r['id'],
            "geometry" : {
                "type": "Point",
                "coordinates": [r['longitude'], r['latitude']]
            },
            "properties" : {
                # Title
                'title': r['title'],
                'title__zh-hant': r['title__zh-hant'],
                'title__zh-hans': r['title__zh-hant'],
                'titleGen': r['titleGen'],
                'titleGen__zh-hant': r['titleGen__zh-hant'],
                'titleGen__zh-hans': r['titleGen__zh-hant'],
                
                # Description
                'description': r['description'],
                'description__zh-hant': r['description__zh-hant'],
                'description__zh-hans': r['description__zh-hant'],
                'descriptionGen': r['descriptionGen'],
                'descriptionGen__zh-hant' : r['descriptionGen__zh-hant'],
                'descriptionGen__zh-hans' : r['descriptionGen__zh-hant'],
                
                # Misc
                'grade' : choice([1,2,3,4,5]),
                
                # Custom
                "graphemes" : r['graphemes'],
                "size" : choice(['small','medium','large']),
                "material" : choice(["painted on cement", "painted on metal", "painted on brick", "painted on tile", "painted over", "acrylic", "metal", "wood", "terrazzo", "stone", "molded cement", "zinc stenciled"]),
                "visibility" : choice(['revenant', 'physical', 'palimpsest', 'uncovering']),
            },
            "addressProperties":{
                "distanceFromPoint" : r["distanceFromPoint"],
                "formattedAddress" : r["formattedAddress"],
                "formattedAddress__zh-hant" : r["formattedAddress__zh-hant"],
                "formattedAddress__zh-hans" : r["formattedAddress__zh-hant"],
                "plusCode" : r["plusCode"],
                "plusCode__zh-hant" : r["plusCode__zh-hant"],
                "plusCode__zh-hans" : r["plusCode__zh-hans"],
                "subPremise" : r["subPremise"],
                "subPremise__zh-hant" : r["subPremise__zh-hant"],
                "subPremise__zh-hans" : r["subPremise__zh-hans"],
                "premise" : r["premise"],
                "premise__zh-hant" : r["premise__zh-hant"],
                "premise__zh-hans" : r["premise__zh-hans"],
                "streetNumber" : r["streetNumber"],
                "streetNumber__zh-hant" : r["streetNumber__zh-hant"],
                "streetNumber__zh-hans" : r["streetNumber__zh-hans"],
                "route" : r["route"],
                "route__zh-hant" : r["route__zh-hant"],
                "route__zh-hans" : r["route__zh-hans"],
                "intersection" : r["intersection"],
                "intersection__zh-hant" : r["intersection__zh-hant"],
                "intersection__zh-hans" : r["intersection__zh-hans"],
                "neighbourhood" : r["neighbourhood"],
                "neighbourhood__zh-hant" : r["neighbourhood__zh-hant"],
                "neighbourhood__zh-hans" : r["neighbourhood__zh-hans"],
                "administrativeAreaLevel1" : r["administrativeAreaLevel1"],
                "administrativeAreaLevel1__zh-hant" : r["administrativeAreaLevel1__zh-hant"],
                "administrativeAreaLevel1__zh-hans" : r["administrativeAreaLevel1__zh-hans"],
                "country" : r["country"],
                "country__zh-hant" : r["country__zh-hant"],
                "country__zh-hans" : r["country__zh-hans"],
                "googlePlaceId" : r["googlePlaceId"],
                "addressGeocoder": r["geocoder"],
                "addressReverseGen": r["reverseGen"],
                "addressForwardGen": r["forwardGen"],
            },
            "layerId" : "7akti7OImSFg",
            "contributorId" : tijptjikId,
            "publisherId" : tijptjikId,
            "isPublished" : True,
            "isIntangible" : False,
            "isVisitable" : True,
            "visitableAsOf" : r['visitableAsOf']
        }
    )

In [ ]:
def default(o):
    if type(o) is date:
        return o.isoformat()

def jsondump(o, f):
    return json.dump(o, f, ensure_ascii=False, indent=4, default=default)

with open(JSONDEST, 'w', encoding='utf-8') as f:
    jsondump(features, f)

# Google Translate Cantonese Experiment

In [ ]:
# Initialize Translation client
def translate(
    contents: str | list[str],
    source_lang : str = "en",
    target_lang : str = "zh-HK",
    project_id: str = "ghostmapper",
    model_id : str = 'NM3b0572414904af9c',
    location : str = 'us-central1',
    print_results: bool = False
) -> google_translate.TranslationServiceClient:
    """Translating text with Google Translation API."""

    client = google_translate.TranslationServiceClient()

    parent = f"projects/{project_id}/locations/{location}"
    model = f"{parent}/models/{model_id}"

    if not type(contents) == list:
        contents = [contents]
    
    # Translate text from English to Hong Kong Chinese with a custom model
    # See https://console.cloud.google.com/translation/models/list?project=ghostmapper 
    response = client.translate_text(
        request={
            "contents": contents,
            "parent": parent,
            "model": model,
            "source_language_code": source_lang,
            "target_language_code": target_lang,
            "mime_type": "text/plain",  # mime types: text/plain, text/html
        }
    )

    # Display the translation for each input text provided
    if print_results:
        print(f'TRANSLATION >>> {source_lang} > {target_lang}')
        for translation in response.translations:
            print(f'{text} --> {translation.translated_text}')

    return [t.translated_text for t in response.translations]

translate_en_to_zhhk = lambda text, print_results: translate(text, "en", "zh-HK", print_results=print_results)
# translate_en_to_zhcn = lambda text, print_results: translate(text, "en", "zh-CN")
# translate_zhhk_to_en = lambda text, print_results: translate(text, "zh-HK", "en")
# translate_zhcn_to_en = lambda text, print_results: translate(text, "zh-CN", "en")
# translate_zhhk_to_zhcn = lambda text, print_results: translate(text, "zh-HK", "zh-CN")


text = 'Hello World, let\'s guide you through the wonderful world of Hong Kong'

translate_en_to_zhhk(text, True)


## Temporary Interpolated Results

In [ ]:
# DESC : Central Pier 5
# DESC : DHL Express Service Point
# DESC : New Treasure Centre

In [ ]:
# "'Strong Hardware'",
#  "'Remove'",
#  "'Songji Daxin Fashion'",
 
#  "'WenJi decoration project' in yellow",
#  "'Ming Hua Noodle House'",
#  "'Hongchang Tea House'",
#  "'He Kee Hardware'",
#  "'Shengli'",
#  "'Xinglu Shiduo'",
#  "'Pan Ji Wood Paint'",
#  "'Telemo Pump'",
#  "'Fourth floor of 'Hanglebie'",
#  "'Oshidahexie'",
#  "'Ya Nguyen'",
#  "'Huang Ti Nong's Western Clothes'",